# 06 — Plugins Deep Dive

Runs the **real** OCR (EasyOCR + multi-orientation + CLAHE), Color (Lab + K-Means + CIEDE2000), and Barcode (9-stage adaptive decode) plugins on actual crops — no mocked evidence. This closes the loop opened in `05_decision_reranking.ipynb`, where evidence was hand-built to match the schema read by `Reranker`; here we verify that schema against what the real plugins actually produce.

Covered in this notebook:

1. Setup and helpers
2. Load the three plugins (`OcrPlugin` needs `easyocr` — wrapped in try/except)
3. Collect real crops that trigger plugin evidence (Detector -> Cropper -> Retriever -> DecisionEngine)
4. OCR deep dive — CLAHE before/after, all rotations side by side, fragment boxes
5. Color deep dive — ROI detection overlay, K-Means palette, ΔE against catalog colors
6. Barcode deep dive — 9-stage adaptive pipeline, stage-by-stage visualization
7. Schema cross-check against `05_decision_reranking.ipynb`'s hand-built evidence
8. Full real fusion — real `PluginResult` through the real `Reranker`
9. Summary


## 1. Environment setup

(identical boilerplate to `04_retrieval_analysis.ipynb` §1 / `05_decision_reranking.ipynb` §1)

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "configs" / "config.yaml").is_file()

In [ ]:
import json
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.core.config import load_config
from src.detection.detector import Detector
from src.detection.cropper import Cropper
from src.retrieval.retriever import Retriever
from src.decision.decision import DecisionEngine
from src.decision.reranker import Reranker
from src.plugins.ocr import OcrPlugin
from src.plugins.color import ColorPlugin
from src.plugins.barcode import BarcodePlugin
from src.models.models import ImageData, RefinementResult, PluginResult

pd.set_option("display.max_colwidth", 80)
%matplotlib inline


def show_image_bgr(ax, image_array_bgr, title=""):
    ax.imshow(cv2.cvtColor(image_array_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def load_image_data(image_path):
    data = np.fromfile(str(image_path), dtype=np.uint8)
    image_array = cv2.imdecode(data, cv2.IMREAD_COLOR)
    height, width = image_array.shape[:2]
    return ImageData(image_id=image_path.stem, source_path=str(image_path),
                      image_array=image_array, width=width, height=height)


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## 2. Config overview

In [ ]:
config = load_config()

print("--- plugins ---")
print(f"plugins.enabled:        {config.plugins.enabled}")
print(f"ocr.enabled:             {config.plugins.ocr.enabled}  rotation_angles={config.plugins.ocr.rotation_angles}")
print(f"color.enabled:           {config.plugins.color.enabled}  n_clusters={config.plugins.color.n_clusters}")
print(f"barcode.enabled:         {config.plugins.barcode.enabled}  preprocessing_enabled={config.plugins.barcode.preprocessing_enabled}")
print(f"\nforce_rules:             {config.plugins.force_rules}")
print(f"rerank.color.references_path: {config.rerank.color.references_path}")

## 3. Load the three plugins

`OcrPlugin.__init__` loads an `easyocr.Reader` immediately when `plugins.ocr.enabled=True` — this can download model weights on first run and is noticeably slower than `ColorPlugin`/`BarcodePlugin` to construct. Wrapped in try/except per the "always guard heavy backends" convention (`04_retrieval_analysis.ipynb` §2, `05_decision_reranking.ipynb` §3).

In [ ]:
try:
    ocr_plugin = OcrPlugin(config)
    print(f"OcrPlugin loaded (enabled={ocr_plugin.is_enabled()}).")
except ImportError as exc:
    print(f"OcrPlugin could not be loaded: {exc}")
    ocr_plugin = None

color_plugin = ColorPlugin(config)
print(f"ColorPlugin loaded (enabled={color_plugin.is_enabled()}).")

barcode_plugin = BarcodePlugin(config)
print(f"BarcodePlugin loaded (enabled={barcode_plugin.is_enabled()}).")

## 4. Collect real crops that trigger plugin evidence

Same upstream chain as `05_decision_reranking.ipynb` §5 (Detector -> Cropper -> Retriever -> `DecisionEngine.decide()`), but scanning a small batch of images and keeping only crops where `needs_plugin=True` — these are exactly the crops `InventoryPipeline` would hand to `PluginManager` at runtime.

In [ ]:
RUN_BUILD_IF_MISSING = True
try:
    retriever = Retriever(config)
    print(f"Retriever loaded. Gallery size: {retriever._index.ntotal} vectors.")
except FileNotFoundError as exc:
    print(f"Retriever could not be loaded: {exc}")
    retriever = None
    if RUN_BUILD_IF_MISSING:
        from src.pipeline.build import BuildPipeline
        build_config = config.model_copy(update={
            "catalog": config.catalog.model_copy(update={"build_metadata": True}),
            "retrieval": config.retrieval.model_copy(update={"build_gallery_index": True}),
        })
        BuildPipeline(build_config).run()
        retriever = Retriever(config)
        print(f"Build complete. Gallery size: {retriever._index.ntotal} vectors.")

decision_engine = DecisionEngine(config)
product_lookup = retriever.get_product if retriever is not None else (lambda pid: None)
reranker = Reranker(config, decision_engine, product_lookup)

In [ ]:
plugin_crops = []  # list of (crop, retrieval_result, decision)

if retriever is not None:
    benchmark_images_dir = config.resolve_path(config.paths.benchmark_images_dir)
    query_dir = config.resolve_path(config.paths.query_dir)
    candidate_dir = benchmark_images_dir if benchmark_images_dir.exists() and any(
        benchmark_images_dir.iterdir()
    ) else query_dir
    available_images = sorted([f for f in candidate_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])

    detector = Detector(config)
    cropper = Cropper(config)
    empty_refinement = lambda image_id: RefinementResult(image_id=image_id, triggered=False, backend="none")

    SCAN_LIMIT = min(15, len(available_images))
    for path in random.sample(available_images, k=SCAN_LIMIT):
        image = load_image_data(path)
        crops = cropper.crop(image, detector.detect(image), empty_refinement(image.image_id))
        for crop in crops:
            retrieval_result = retriever.retrieve(crop)
            decision = decision_engine.decide(retrieval_result)
            if decision.needs_plugin:
                plugin_crops.append((crop, retrieval_result, decision))
        if len(plugin_crops) >= 6:
            break

    print(f"Scanned up to {SCAN_LIMIT} image(s), found {len(plugin_crops)} crop(s) with needs_plugin=True.")
    for crop, _, decision in plugin_crops:
        print(f"  crop_id={crop.crop_id}  trigger_reasons={sorted(decision.trigger_reasons)}  "
              f"forced_plugins={sorted(decision.forced_plugins)}")
else:
    print("Retriever not available — skip.")

assert plugin_crops, "No plugin-triggering crops found in this sample — try increasing SCAN_LIMIT or a different image set."
demo_crop, demo_retrieval_result, demo_decision = plugin_crops[0]
print(f"\nUsing crop_id='{demo_crop.crop_id}' as the running example for sections 4-6.")

## 5. OCR deep dive

Visualizes what `OcrPlugin._preprocess()` (adaptive upscale + CLAHE on the L channel) does to the raw crop, then runs every configured rotation and compares them using the plugin's own `orientation_scores` diagnostics.

In [ ]:
if ocr_plugin is not None:
    raw = demo_crop.raw_image_array
    preprocessed = ocr_plugin._preprocess(raw)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    show_image_bgr(axes[0], raw, title="raw_image_array")
    show_image_bgr(axes[1], preprocessed, title="after _preprocess()\n(upscale + CLAHE on L channel)")
    plt.tight_layout()
    plt.show()
else:
    print("OcrPlugin not available — skip.")

In [ ]:
if ocr_plugin is not None:
    rotations = ocr_plugin._get_rotations()
    fig, axes = plt.subplots(1, len(rotations), figsize=(3.5 * len(rotations), 4))
    if len(rotations) == 1:
        axes = [axes]
    for ax, rotation in zip(axes, rotations):
        rotated = ocr_plugin._rotate(preprocessed, rotation)
        show_image_bgr(ax, rotated, title=f"rotation={rotation}")
    plt.tight_layout()
    plt.show()
else:
    print("OcrPlugin not available — skip.")

In [ ]:
if ocr_plugin is not None:
    ocr_output = ocr_plugin.run(demo_crop)

    sel_text = ocr_output["text"]
    sel_rotation = ocr_output["rotation"]
    print(f"Selected text: '{sel_text}'  (rotation={sel_rotation})")
    print(f"score={ocr_output["score"]:.4f}  confidence={ocr_output["confidence"]:.4f}  "
          f"information_score={ocr_output["information_score"]:.4f}")
    print(f"orientation_ambiguous={ocr_output["orientation_ambiguous"]}  "
          f"orientation_candidate_count={ocr_output["orientation_candidate_count"]}")

    orientation_df = pd.DataFrame(ocr_output["orientation_scores"])[
        ["rotation", "score", "confidence", "information_score", "useful_text_length",
         "useful_fragment_count", "fragment_count", "text"]
    ]
    display(orientation_df)
else:
    ocr_output = None
    print("OcrPlugin not available — skip.")

In [ ]:
if ocr_plugin is not None and ocr_output["ocr_boxes"]:
    fig, ax = plt.subplots(figsize=(6, 6))
    winning_rotation = ocr_output["rotation"]
    display_image = ocr_plugin._rotate(preprocessed, winning_rotation)
    show_image_bgr(ax, display_image, title=f"OCR fragments @ rotation={winning_rotation}")

    for box in ocr_output["ocr_boxes"]:
        bbox = np.array(box["bbox"])
        if bbox.size == 0:
            continue
        polygon = plt.Polygon(bbox, closed=True, fill=False, edgecolor="#55A868", linewidth=1.5)
        ax.add_patch(polygon)
        label = f"{box["text"]} ({box["confidence"]:.2f})"
        ax.text(bbox[0, 0], bbox[0, 1] - 4, label, color="#55A868", fontsize=8)
    plt.tight_layout()
    plt.show()
elif ocr_plugin is not None:
    print("No OCR fragments detected on the winning orientation for this crop.")

## 6. Color deep dive

Visualizes `ColorPlugin._extract_powder_roi()`'s detected region on the raw crop, the K-Means palette, and the resulting CIEDE2000 distance to every catalog color reference (reusing `Reranker._delta_e_2000`, the exact function used at runtime — no reimplementation).

In [ ]:
color_output = color_plugin.run(demo_crop)

roi_info_05 = color_output["roi"]
print(f"dominant_color: {color_output["dominant_color"]}  representative_lab: {color_output["representative_lab"]}")
print(f"ROI method: {roi_info_05["method"]}  score={roi_info_05["score"]:.3f}  "
      f"area_ratio={roi_info_05["area_ratio"]:.3f}  fallback_used={roi_info_05["fallback_used"]}")
print(f"dominant_selection_method: {color_output["debug"]["dominant_selection_method"]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

x, y, w, h = color_output["roi"]["bbox"]
overlay = demo_crop.raw_image_array.copy()
cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 200, 0), 3)
roi_method_label = color_output["roi"]["method"]
show_image_bgr(axes[0], overlay, title=f"Detected ROI ({roi_method_label})")

palette_hex = color_output["palette"]
percentages = color_output["percentages"]
axes[1].bar(range(len(palette_hex)), percentages, color=palette_hex, edgecolor="black")
axes[1].set_xticks(range(len(palette_hex)))
axes[1].set_xticklabels(palette_hex, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Cluster pixel share")
axes[1].set_title("K-Means palette (ordered by weight)")
plt.tight_layout()
plt.show()

In [ ]:
color_reference_path = config.resolve_path(config.rerank.color.references_path)
if color_reference_path.is_file():
    with open(color_reference_path, "r", encoding="utf-8") as f:
        color_reference_raw = json.load(f)

    query_lab = color_output["representative_lab"]
    rows = []
    for code_name, entry in color_reference_raw.items():
        r, g, b = entry["rgb"]
        bgr = np.array([[[b, g, r]]], dtype=np.uint8)
        ref_lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)[0, 0].astype(float).tolist()
        delta_e = Reranker._delta_e_2000(query_lab, ref_lab)
        rows.append({"color_code": code_name, "hex": entry.get("hex", ""), "delta_e_2000": round(delta_e, 3)})

    delta_e_df = pd.DataFrame(rows).sort_values("delta_e_2000").reset_index(drop=True)
    print(f"query representative_lab: {query_lab}")
    display(delta_e_df)
    print(f"delta_e_strong={config.rerank.color.delta_e_strong}  delta_e_weak={config.rerank.color.delta_e_weak}")
else:
    print(f"Color reference file not found at {color_reference_path} — skip.")

## 7. Barcode deep dive

Walks `BarcodePlugin`'s adaptive 9-stage pipeline by calling its private stage methods directly on the demo crop's grayscale image, mirroring the same order `_decode_with_pipeline()` follows internally.

In [ ]:
gray = cv2.cvtColor(demo_crop.raw_image_array, cv2.COLOR_BGR2GRAY)
cfg = config.plugins.barcode

has_edges = barcode_plugin._has_sufficient_edges(gray)
print(f"Stage -1 (presence pre-check): sufficient edge density = {has_edges}")

region = barcode_plugin._detect_barcode_region(gray) if cfg.presence_check_enabled else None
print(f"Stage 1 (region detection): {region}")

In [ ]:
stage_images = {"0_raw_gray": gray}

working = gray
if region is not None:
    working = barcode_plugin._crop_to_region(working, region["bbox"])
    stage_images["1_cropped_to_region"] = working

if cfg.deskew_enabled:
    angle = region["angle"] if region is not None else barcode_plugin._estimate_skew_angle(working)
    if abs(angle) > 0.5:
        working = barcode_plugin._rotate_image(working, angle)
    stage_images[f"2_deskewed_angle_{angle:.1f}"] = working

if cfg.upscale_enabled:
    working = barcode_plugin._upscale(working)
    stage_images["3_upscaled"] = working

if cfg.clahe_enabled:
    working = barcode_plugin._apply_clahe(working)
    stage_images["4_clahe"] = working

if cfg.adaptive_threshold_enabled:
    stage_images["5_adaptive_threshold"] = barcode_plugin._adaptive_threshold(working)

if cfg.denoise_enabled:
    working = barcode_plugin._denoise(working)
    stage_images["6_denoised"] = working

if cfg.sharpen_enabled:
    stage_images["7_sharpened"] = barcode_plugin._sharpen(working)

n = len(stage_images)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.5))
if n == 1:
    axes = [axes]
for ax, (title, img) in zip(axes, stage_images.items()):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
barcode_output = barcode_plugin.run(demo_crop)

stage_succeeded = barcode_output["preprocessing_stage"]
print(f"preprocessing_stage that succeeded: '{stage_succeeded}'")
print(f"confidence: {barcode_output["confidence"]:.4f}")
if barcode_output["barcodes"]:
    display(pd.DataFrame(barcode_output["barcodes"]))
else:
    print("No barcode decoded for this crop (expected for most crops — barcode is a force-rule plugin).")

## 8. Schema cross-check against `05_decision_reranking.ipynb`

Confirms the real plugin output shape matches what `05_decision_reranking.ipynb` §9 hand-built and what `Reranker` actually reads.

In [ ]:
checks = []

if ocr_output is not None:
    checks.append(("ocr", "orientation_candidates is a list", isinstance(ocr_output.get("orientation_candidates"), list)))
    if ocr_output["orientation_candidates"]:
        oc0 = ocr_output["orientation_candidates"][0]
        for key in ("rotation", "text", "score", "confidence", "information_score"):
            checks.append(("ocr", f"orientation_candidates[0] has '{key}'", key in oc0))

checks.append(("color", "representative_lab is [L, a, b]",
                isinstance(color_output.get("representative_lab"), list) and len(color_output["representative_lab"]) == 3))
checks.append(("color", "'confidence' key present in evidence dict", "confidence" in color_output))

checks.append(("barcode", "barcodes is a list", isinstance(barcode_output.get("barcodes"), list)))
checks.append(("barcode", "'confidence' key present", "confidence" in barcode_output))

display(pd.DataFrame(checks, columns=["plugin", "check", "passed"]))

> **Finding:** `ColorPlugin.run()` does **not** return a top-level `"confidence"` key (unlike OCR and Barcode) — only `dominant_color`, `dominant_rgb`, `representative_lab`, `palette`, `percentages`, `roi`, `debug`, `latency_ms`. `Reranker._extract_plugin_confidences()` looks for `evidence["color"].get("confidence")` (falling back to `.get("plugin_confidence")`, then a top-level `plugin_result.plugin_confidence` dict which `PluginManager` never sets either) — so `plugin_confidence["color"]` resolves to `0.0` for every real run today. Since `_effective_boost()` multiplies `match_strength × plugin_confidence`, this means **color evidence currently contributes a zero boost in the live pipeline**, even when `color_match_strength` is high and `rerank.color.weight = 0.20`. This is worth flagging as a candidate bug/gap for the codebase, not something to route around in this notebook.

## 9. Full real fusion — real `PluginResult` through the real `Reranker`

Same shape as `05_decision_reranking.ipynb`'s case studies, but every input is now real: real crop, real retrieval candidates, real plugin evidence (`PluginManager`-equivalent — running every plugin whose reason is in `demo_decision.trigger_reasons`, matching `only_forced = "force" in decision.trigger_reasons` from `src/plugins/manager.py`).

In [ ]:
only_forced = "force" in demo_decision.trigger_reasons
plugins_to_run = []
for plugin in ([ocr_plugin, color_plugin, barcode_plugin] if ocr_plugin is not None else [color_plugin, barcode_plugin]):
    if plugin is None or not plugin.is_enabled():
        continue
    if only_forced and plugin.name not in demo_decision.forced_plugins:
        continue
    plugins_to_run.append(plugin)

print(f"trigger_reasons={sorted(demo_decision.trigger_reasons)}  only_forced={only_forced}")
print(f"Plugins that would run: {[p.name for p in plugins_to_run]}")

evidence = {}
executed = []
for plugin in plugins_to_run:
    evidence[plugin.name] = plugin.run(demo_crop)
    executed.append(plugin.name)

real_plugin_result = PluginResult(crop_id=demo_crop.crop_id, executed_plugins=executed, evidence=evidence)

print("\nPreliminary decision (visual retrieval only):")
print(f"  product_id={demo_decision.product_id}  status={demo_decision.status}  "
      f"final_confidence={demo_decision.final_confidence:.3f}")

final_real_decision = reranker.rerank(demo_retrieval_result, real_plugin_result)
print("\nFinal decision (after real evidence fusion):")
print(f"  product_id={final_real_decision.product_id}  status={final_real_decision.status}  "
      f"final_confidence={final_real_decision.final_confidence:.3f}")
print(f"  reason: {final_real_decision.reason}")

display(pd.DataFrame(final_real_decision.rerank_debug["candidates"])[
    ["product_id", "base_similarity", "barcode_boost", "ocr_boost", "color_boost", "adjusted_score"]
])

In [ ]:
if len(plugin_crops) > 1:
    batch_rows = []
    for crop, retrieval_result, decision in plugin_crops:
        only_forced_b = "force" in decision.trigger_reasons
        evidence_b = {}
        for plugin in ([ocr_plugin, color_plugin, barcode_plugin] if ocr_plugin is not None else [color_plugin, barcode_plugin]):
            if plugin is None or not plugin.is_enabled():
                continue
            if only_forced_b and plugin.name not in decision.forced_plugins:
                continue
            evidence_b[plugin.name] = plugin.run(crop)

        plugin_result_b = PluginResult(crop_id=crop.crop_id, executed_plugins=list(evidence_b.keys()), evidence=evidence_b)
        final_b = reranker.rerank(retrieval_result, plugin_result_b)

        batch_rows.append({
            "crop_id": crop.crop_id,
            "trigger_reasons": sorted(decision.trigger_reasons),
            "status_before": decision.status,
            "status_after": final_b.status,
            "confidence_before": round(decision.final_confidence, 3),
            "confidence_after": round(final_b.final_confidence, 3),
            "product_switched": decision.product_id != final_b.product_id,
        })

    display(pd.DataFrame(batch_rows))
else:
    print("Only one plugin-triggering crop found in this sample — batch comparison skipped.")

## 10. Summary

In [ ]:
summary = {
    "Plugin-triggering crops found (\u00a74)": len(plugin_crops),
    "OCR selected text (\u00a75)": ocr_output["text"] if ocr_output is not None else "n/a (OcrPlugin unavailable)",
    "Color dominant hex (\u00a76)": color_output["dominant_color"],
    "Barcode preprocessing stage that succeeded (\u00a77)": barcode_output["preprocessing_stage"],
    "Real fusion status change (\u00a79)": f"{demo_decision.status} -> {final_real_decision.status}",
}
pd.DataFrame([summary]).T.rename(columns={0: "Value"})

---
**Quick observations (fill in after running on real data):**

- In §5, did the OCR text extracted actually match text you can read on the crop image? Compare against the fragment box overlay.
- In §6, does the detected ROI (green box) actually land on the product's powder/packaging region, or is it falling back to `lower_center_fallback`/`center`? That's a strong signal for whether `roi_enabled` heuristics need tuning for this product line.
- In §7, which stage typically succeeds for this dataset's crops — raw, deskew, upscale, or one of the later stages? A distribution over many crops would tell you whether the more expensive stages (denoise, sharpen) are pulling their weight.
- In §8, the zero `color` confidence finding — does this match what you observed in `05_decision_reranking.ipynb` §11 (the confusable-pair case study), where `color_boost` might have been silently zero for reasons unrelated to the ΔE calculation itself?
- In §9's batch table, how often does `status_after` differ from `status_before`? That's the real-world version of the "Evidence Fusion accuracy delta (+0.224)" figure the README reports.

**Next notebook:** `07_end_to_end_pipeline.ipynb` — running `InventoryPipeline.run_with_trace()` on full images, visualizing every stage together, and exporting JSON/CSV/annotated results.